# Karabo-Style KNN Synthetic Data Experiment  
## Final single-notebook version: keep all usable columns

### Requirement update

This notebook follows Karabo's categorical KNN experiment style, but it has been updated for the final methodology:

1. **Do not remove columns just because they are constant, high-cardinality, or low-signal.**
2. **Keep all usable columns as they are.**
3. **Drop only columns that are fully empty or fully missing.**
4. **Work on both categorical columns and numerical columns.**
5. **Keep all outputs inside this notebook.**
6. **Do not generate multiple external CSV/ZIP result files.**

The notebook performs:

- Dataset loading
- Empty-column detection and removal only for fully empty columns
- Categorical and numerical validation
- KNN-based synthetic generation using:
  - categorical-only distance
  - numerical-only distance
  - mixed categorical + numerical distance
- Validation of synthetic data quality
- Utility testing using the target column when available
- Privacy/duplicate risk checks
- Sensitivity testing across KNN settings

## Step 1: Import libraries and configure notebook

The configuration below controls runtime and output behaviour.

Important note:  
`MAX_MODEL_ROWS` and `SYNTHETIC_ROWS` are row-level runtime controls only. They do **not** remove columns.  
All non-empty columns are retained in the methodology.

In [9]:
import warnings
warnings.filterwarnings("ignore")

import os
import math
import itertools
import numpy as np
import pandas as pd

from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

try:
    from scipy.stats import ks_2samp, chi2_contingency
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

RANDOM_STATE = 42
DATA_PATH = "data.csv"

# Runtime controls.
# These keep the notebook practical while preserving every usable column.
MAX_MODEL_ROWS = 5000
SYNTHETIC_ROWS = 2000
VALIDATION_SAMPLE_ROWS = 3000

# KNN settings
DEFAULT_K = 5
MIXED_CATEGORICAL_WEIGHT = 0.50
MIXED_NUMERICAL_WEIGHT = 0.50

# Sensitivity controls
RUN_SENSITIVITY = False  # Set to True when you want to run the optional K sensitivity loop.
SENSITIVITY_K_VALUES = [3, 5, 10]
SENSITIVITY_SYNTHETIC_ROWS = 500

# Final solution rule:
# No multiple files are exported. Results are displayed inside the notebook.
EXPORT_RESULTS = False

np.random.seed(RANDOM_STATE)

print("Configuration loaded.")

Configuration loaded.


## Step 2: Load the dataset

The notebook expects the CSV file to be in the same folder as the notebook.

If the file name is different, update `DATA_PATH` in Step 1.

In [10]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the CSV in the same folder as this notebook "
        "or update DATA_PATH in Step 1."
    )

raw_df = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", raw_df.shape)
display(raw_df.head())

Raw dataset shape: (100000, 40)


,Age,Age_Group,Age_Group5,Age_Group3,Gender,Deceased_Indicator_YN,Marital_Status_YN,Director_YN,Entrepreneur_YN,Email_YN,...,Pay_Type,Number_Of_Business_Partners,ID_Valid_YN,Province,Town,Suburb,Vehicleowner_YN,Number_Of_Vehicles,Months_Since_Most_Recent_Vehicle_Purchase,target
0,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,Unknown,01. 0,1,NaN,NaN,NaN,N,0,00. NA,0
1,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,Unknown,01. 0,1,NaN,NaN,NaN,N,0,00. NA,1
2,19,02. 18-29,02. 18-24,02. 18-20,Male,N,N,N,N,N,...,Unknown,01. 0,1,NaN,NaN,NaN,N,0,00. NA,1
3,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,Unknown,01. 0,1,NaN,NaN,NaN,N,0,00. NA,0
4,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,Unknown,01. 0,1,NaN,NaN,NaN,N,0,00. NA,1


## Step 3: Standardise blank values and drop only fully empty columns

As per the final requirement, we do **not** remove constant columns or low-signal columns.

We only remove columns where all values are missing or blank.

Examples of columns that will be dropped:

- Entire column is `NaN`
- Entire column is blank string
- Entire column is whitespace
- Entire column becomes missing after blank cleanup

Everything else is kept.

In [11]:
df = raw_df.copy()

# Convert blank strings and whitespace-only strings to NaN for empty-column detection.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

def is_fully_empty(series: pd.Series) -> bool:
    return series.isna().all()

empty_columns = [col for col in df.columns if is_fully_empty(df[col])]

# Drop only fully empty columns.
df = df.drop(columns=empty_columns)

print("Original column count:", raw_df.shape[1])
print("Dropped fully empty/missing columns:", len(empty_columns))
print(empty_columns)
print("Final usable column count:", df.shape[1])
print("Final dataset shape:", df.shape)

Original column count: 40
Dropped fully empty/missing columns: 4
['IsParent', 'Province', 'Town', 'Suburb']
Final usable column count: 36
Final dataset shape: (100000, 36)


## Step 4: Detect target, categorical columns, and numerical columns

The notebook automatically checks whether a `target` column exists.

Rules used:

- `target` is kept separately for splitting, validation, and utility testing.
- Categorical columns are object/string/category/bool columns.
- Numerical columns are integer/float columns.
- Constant columns are **not removed**.
- High-cardinality columns are **not removed**.
- Columns are only flagged in validation so we understand their behaviour.

In [12]:
target_col = "target" if "target" in df.columns else None

feature_cols = [c for c in df.columns if c != target_col]

categorical_cols = [
    c for c in feature_cols
    if pd.api.types.is_object_dtype(df[c])
    or pd.api.types.is_string_dtype(df[c])
    or pd.api.types.is_categorical_dtype(df[c])
    or pd.api.types.is_bool_dtype(df[c])
]

numerical_cols = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

print("Target column:", target_col)
print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))
print("Total feature columns kept:", len(feature_cols))

display(pd.DataFrame({
    "column_type": ["categorical", "numerical", "target"],
    "count": [len(categorical_cols), len(numerical_cols), 1 if target_col else 0]
}))

Target column: target
Categorical columns: 31
Numerical columns: 4
Total feature columns kept: 35


,column_type,count
0,categorical,31
1,numerical,4
2,target,1


## Step 5: Column-level validation

This step creates a full column validation table.

The purpose is not to remove columns.  
The purpose is to understand each column before modelling.

Validation checks:

- Data type
- Missing count
- Missing rate
- Unique values
- Constant column flag
- High-cardinality flag
- Role in the experiment

In [13]:
validation_rows = []

for col in df.columns:
    if col == target_col:
        role = "target"
    elif col in categorical_cols:
        role = "categorical_feature"
    elif col in numerical_cols:
        role = "numerical_feature"
    else:
        role = "other_feature"

    missing_count = int(df[col].isna().sum())
    missing_rate = missing_count / len(df)
    unique_count = int(df[col].nunique(dropna=False))
    non_missing_unique_count = int(df[col].nunique(dropna=True))

    validation_rows.append({
        "column": col,
        "role": role,
        "dtype": str(df[col].dtype),
        "missing_count": missing_count,
        "missing_rate": round(missing_rate, 6),
        "unique_count_including_missing": unique_count,
        "unique_count_excluding_missing": non_missing_unique_count,
        "is_constant": non_missing_unique_count <= 1,
        "is_high_cardinality": (non_missing_unique_count > 50) if role == "categorical_feature" else False,
        "action": "kept" if col not in empty_columns else "dropped_empty"
    })

column_validation = pd.DataFrame(validation_rows)

display(column_validation)

display(
    column_validation
    .groupby(["role", "is_constant"], dropna=False)
    .size()
    .reset_index(name="column_count")
)

,column,role,dtype,missing_count,missing_rate,unique_count_including_missing,unique_count_excluding_missing,is_constant,is_high_cardinality,action
0,Age,numerical_feature,int64,0,0.0,10,10,False,False,kept
1,Age_Group,categorical_feature,str,0,0.0,2,2,False,False,kept
2,Age_Group5,categorical_feature,str,0,0.0,2,2,False,False,kept
3,Age_Group3,categorical_feature,str,0,0.0,2,2,False,False,kept
4,Gender,categorical_feature,str,0,0.0,2,2,False,False,kept
5,Deceased_Indicator_YN,categorical_feature,str,0,0.0,2,2,False,False,kept
6,Marital_Status_YN,categorical_feature,str,0,0.0,1,1,True,False,kept
7,Director_YN,categorical_feature,str,0,0.0,1,1,True,False,kept
8,Entrepreneur_YN,categorical_feature,str,0,0.0,1,1,True,False,kept
9,Email_YN,categorical_feature,str,0,0.0,1,1,True,False,kept


,role,is_constant,column_count
0,categorical_feature,False,5
1,categorical_feature,True,26
2,numerical_feature,False,1
3,numerical_feature,True,3
4,target,False,1


## Step 6: Prepare data without dropping usable columns

Preparation rules:

### Categorical columns
- Missing values are filled with `__MISSING__`
- Values are treated as strings
- Columns are kept even if they are constant

### Numerical columns
- Missing values are filled using the training median
- Columns are kept even if they are constant
- Numerical features are scaled for KNN distance calculation

### Target column
- Kept for stratified split and utility validation
- Not used as a normal feature in KNN distance calculation

In [14]:
prepared_df = df.copy()

# Categorical missing treatment.
for col in categorical_cols:
    prepared_df[col] = (
        prepared_df[col]
        .astype("object")
        .where(prepared_df[col].notna(), "__MISSING__")
        .astype(str)
    )

# Numerical conversion and missing treatment will be fitted on train only after splitting.
for col in numerical_cols:
    prepared_df[col] = pd.to_numeric(prepared_df[col], errors="coerce")

# Clean target if available.
if target_col is not None:
    # Keep original target values but ensure missing targets are not used in supervised utility.
    print("Target value counts:")
    display(prepared_df[target_col].value_counts(dropna=False).to_frame("count"))

display(prepared_df.head())

Target value counts:


,count
target,
1,50191
0,49809


,Age,Age_Group,Age_Group5,Age_Group3,Gender,Deceased_Indicator_YN,Marital_Status_YN,Director_YN,Entrepreneur_YN,Email_YN,...,Salary_Prediction,Salary_Prediction_Highvalue,Social_Class,Pay_Type,Number_Of_Business_Partners,ID_Valid_YN,Vehicleowner_YN,Number_Of_Vehicles,Months_Since_Most_Recent_Vehicle_Purchase,target
0,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,00. NA,00. NA,00. Unknown,Unknown,01. 0,1,N,0,00. NA,0
1,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,00. NA,00. NA,00. Unknown,Unknown,01. 0,1,N,0,00. NA,1
2,19,02. 18-29,02. 18-24,02. 18-20,Male,N,N,N,N,N,...,00. NA,00. NA,00. Unknown,Unknown,01. 0,1,N,0,00. NA,1
3,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,00. NA,00. NA,00. Unknown,Unknown,01. 0,1,N,0,00. NA,0
4,53,00. Deceased,00. Deceased,00. Deceased,Male,Y,N,N,N,N,...,00. NA,00. NA,00. Unknown,Unknown,01. 0,1,N,0,00. NA,1


## Step 7: Train-validation split

Synthetic data is generated from the training set only.

The validation set is used to check whether the synthetic data preserves patterns that generalise beyond the training data.

If the target column exists and has enough classes, stratified splitting is used.

In [15]:
if target_col is not None and prepared_df[target_col].nunique(dropna=True) > 1:
    stratify_values = prepared_df[target_col]
else:
    stratify_values = None

train_df, valid_df = train_test_split(
    prepared_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=stratify_values
)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)

if target_col is not None:
    print("Train target distribution:")
    display(train_df[target_col].value_counts(normalize=True, dropna=False).to_frame("train_rate"))
    print("Validation target distribution:")
    display(valid_df[target_col].value_counts(normalize=True, dropna=False).to_frame("valid_rate"))

Train shape: (80000, 36)
Validation shape: (20000, 36)
Train target distribution:


,train_rate
target,
1,0.501912
0,0.498088


Validation target distribution:


,valid_rate
target,
1,0.5019
0,0.4981


## Step 8: Row sampling for runtime control

KNN can become heavy on large datasets.  
So this notebook uses row sampling for KNN fitting and validation.

This does **not** remove columns.  
Every usable column remains available in the experiment.

In [16]:
def stratified_or_random_sample(data, n, target_col=None, random_state=42):
    if len(data) <= n:
        return data.copy()

    if target_col is not None and data[target_col].nunique(dropna=True) > 1:
        return (
            data.groupby(target_col, group_keys=False)
            .apply(lambda x: x.sample(
                n=max(1, int(round(n * len(x) / len(data)))),
                random_state=random_state
            ))
            .sample(frac=1, random_state=random_state)
            .head(n)
            .copy()
        )

    return data.sample(n=n, random_state=random_state).copy()

train_model_df = stratified_or_random_sample(
    train_df,
    n=MAX_MODEL_ROWS,
    target_col=target_col,
    random_state=RANDOM_STATE
)

valid_eval_df = stratified_or_random_sample(
    valid_df,
    n=VALIDATION_SAMPLE_ROWS,
    target_col=target_col,
    random_state=RANDOM_STATE
)

print("KNN training sample shape:", train_model_df.shape)
print("Validation evaluation sample shape:", valid_eval_df.shape)

KNN training sample shape: (5000, 35)
Validation evaluation sample shape: (3000, 35)


## Step 9: Impute numerical values using training medians

Numerical imputation must be fitted only on the training sample to avoid leakage.

All numerical columns are kept.

In [17]:
train_model_df = train_model_df.copy()
valid_eval_df = valid_eval_df.copy()

numeric_medians = {}
numeric_min = {}
numeric_max = {}
integer_like_cols = []

for col in numerical_cols:
    median_value = train_model_df[col].median()
    if pd.isna(median_value):
        median_value = 0

    numeric_medians[col] = median_value
    numeric_min[col] = train_model_df[col].min()
    numeric_max[col] = train_model_df[col].max()

    # Detect integer-like columns so synthetic values can be rounded back.
    non_missing_values = train_model_df[col].dropna()
    is_integer_like = False
    if len(non_missing_values) > 0:
        is_integer_like = np.all(np.isclose(non_missing_values, np.round(non_missing_values)))
    if is_integer_like:
        integer_like_cols.append(col)

    train_model_df[col] = train_model_df[col].fillna(median_value)
    valid_eval_df[col] = valid_eval_df[col].fillna(median_value)

print("Numerical medians fitted from train sample:")
display(pd.DataFrame({"column": list(numeric_medians.keys()), "median": list(numeric_medians.values())}))

print("Integer-like numerical columns:")
print(integer_like_cols)

Numerical medians fitted from train sample:


,column,median
0,Age,19.0
1,LSM,0.0
2,ID_Valid_YN,1.0
3,Number_Of_Vehicles,0.0


Integer-like numerical columns:
['Age', 'LSM', 'ID_Valid_YN', 'Number_Of_Vehicles']


## Step 10: Build feature matrices for KNN

We create three KNN views:

1. **Categorical-only view**  
   Uses one-hot encoded categorical columns.

2. **Numerical-only view**  
   Uses robust-scaled numerical columns.

3. **Mixed view**  
   Uses both categorical and numerical columns.

All usable columns remain in the final synthetic output.  
The view only controls how neighbours are found.

In [18]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_knn_matrix(train_data, mode="mixed", cat_weight=0.5, num_weight=0.5):
    fitted = {}
    parts = []

    use_cat = mode in ["categorical", "mixed"] and len(categorical_cols) > 0
    use_num = mode in ["numerical", "mixed"] and len(numerical_cols) > 0

    # Fallback if a selected mode has no available columns.
    if not use_cat and not use_num:
        if len(categorical_cols) > 0:
            use_cat = True
        elif len(numerical_cols) > 0:
            use_num = True
        else:
            raise ValueError("No usable feature columns available for KNN.")

    if use_cat:
        ohe = make_one_hot_encoder()
        cat_matrix = ohe.fit_transform(train_data[categorical_cols].astype(str))
        cat_matrix = np.asarray(cat_matrix, dtype=float)
        if mode == "mixed":
            cat_matrix = cat_matrix * cat_weight
        fitted["ohe"] = ohe
        parts.append(cat_matrix)

    if use_num:
        scaler = RobustScaler()
        num_matrix = scaler.fit_transform(train_data[numerical_cols])
        num_matrix = np.asarray(num_matrix, dtype=float)
        if mode == "mixed":
            num_matrix = num_matrix * num_weight
        fitted["scaler"] = scaler
        parts.append(num_matrix)

    X = np.hstack(parts)
    fitted["mode"] = mode
    fitted["cat_weight"] = cat_weight
    fitted["num_weight"] = num_weight

    return X, fitted

X_cat, fit_cat = build_knn_matrix(train_model_df, mode="categorical")
X_num, fit_num = build_knn_matrix(train_model_df, mode="numerical")
X_mix, fit_mix = build_knn_matrix(
    train_model_df,
    mode="mixed",
    cat_weight=MIXED_CATEGORICAL_WEIGHT,
    num_weight=MIXED_NUMERICAL_WEIGHT
)

print("Categorical KNN matrix:", X_cat.shape)
print("Numerical KNN matrix:", X_num.shape)
print("Mixed KNN matrix:", X_mix.shape)

Categorical KNN matrix: (5000, 36)
Numerical KNN matrix: (5000, 4)
Mixed KNN matrix: (5000, 40)


## Step 11: KNN synthetic data generation function

For every synthetic row:

1. Select a base row.
2. Find its nearest neighbours using the selected KNN view.
3. Generate categorical values by weighted sampling from neighbour categories.
4. Generate numerical values by interpolation between the base row and a selected neighbour.
5. Clip numerical values to training min/max.
6. Round integer-like numerical columns.
7. Sample the target from neighbours when the target exists.

The final synthetic dataframe keeps the same non-empty columns as the cleaned input.

In [19]:
def generate_synthetic_knn(
    train_data,
    X,
    mode_name,
    n_rows=5000,
    k=5,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    n_train = len(train_data)
    if n_train < 2:
        raise ValueError("Need at least 2 training rows for KNN synthesis.")

    n_neighbors = min(k + 1, n_train)

    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean", algorithm="brute", n_jobs=-1)
    nn.fit(X)

    base_indices = rng.integers(0, n_train, size=n_rows)
    distances, neighbour_indices = nn.kneighbors(X[base_indices])

    rows = []
    train_reset = train_data.reset_index(drop=True)

    for i in range(n_rows):
        base_idx = int(base_indices[i])
        neighs = neighbour_indices[i].astype(int)
        dists = distances[i].astype(float)

        # Remove the base row itself from the neighbour pool when possible.
        mask = neighs != base_idx
        if mask.sum() > 0:
            neighs = neighs[mask]
            dists = dists[mask]

        if len(neighs) == 0:
            neighs = np.array([base_idx])
            dists = np.array([0.0])

        weights = 1.0 / (dists + 1e-6)
        weights = weights / weights.sum()

        selected_neighbour = int(rng.choice(neighs, p=weights))

        base_row = train_reset.iloc[base_idx]
        neighbour_row = train_reset.iloc[selected_neighbour]
        neighbour_pool = train_reset.iloc[neighs]

        synthetic_row = {}

        # Categorical generation.
        for col in categorical_cols:
            values = neighbour_pool[col].astype(str).to_numpy()
            synthetic_row[col] = rng.choice(values, p=weights)

        # Numerical generation.
        for col in numerical_cols:
            base_value = float(base_row[col])
            neighbour_value = float(neighbour_row[col])
            lam = rng.random()

            synthetic_value = base_value + lam * (neighbour_value - base_value)

            # Clip to training range.
            lower = numeric_min.get(col, np.nan)
            upper = numeric_max.get(col, np.nan)

            if pd.notna(lower) and pd.notna(upper):
                synthetic_value = float(np.clip(synthetic_value, lower, upper))

            if col in integer_like_cols:
                synthetic_value = int(round(synthetic_value))

            synthetic_row[col] = synthetic_value

        # Target generation, if available.
        if target_col is not None:
            target_values = neighbour_pool[target_col].to_numpy()
            synthetic_row[target_col] = rng.choice(target_values, p=weights)

        rows.append(synthetic_row)

    synthetic_df = pd.DataFrame(rows)

    # Preserve original cleaned column order.
    final_columns = [c for c in df.columns if c in synthetic_df.columns]
    synthetic_df = synthetic_df[final_columns]

    synthetic_df.attrs["mode"] = mode_name
    synthetic_df.attrs["k"] = k

    return synthetic_df

## Step 12: Generate synthetic data using categorical, numerical, and mixed KNN

The mixed version is the recommended final synthetic output because it uses both categorical and numerical information.

However, categorical-only and numerical-only versions are also generated for comparison.

In [20]:
synthetic_categorical = generate_synthetic_knn(
    train_model_df,
    X_cat,
    mode_name="categorical_only",
    n_rows=SYNTHETIC_ROWS,
    k=DEFAULT_K,
    random_state=RANDOM_STATE
)

synthetic_numerical = generate_synthetic_knn(
    train_model_df,
    X_num,
    mode_name="numerical_only",
    n_rows=SYNTHETIC_ROWS,
    k=DEFAULT_K,
    random_state=RANDOM_STATE + 1
)

synthetic_mixed = generate_synthetic_knn(
    train_model_df,
    X_mix,
    mode_name="mixed_categorical_numerical",
    n_rows=SYNTHETIC_ROWS,
    k=DEFAULT_K,
    random_state=RANDOM_STATE + 2
)

print("Synthetic categorical-only shape:", synthetic_categorical.shape)
print("Synthetic numerical-only shape:", synthetic_numerical.shape)
print("Synthetic mixed shape:", synthetic_mixed.shape)

display(synthetic_mixed.head())

KeyError: 'target'

## Step 13: Categorical validation

Categorical validation checks whether synthetic data preserves category-level behaviour.

Metrics:

- Real category count
- Synthetic category count
- Category coverage
- New synthetic category count
- Mean distribution drift
- Maximum distribution drift

Distribution drift is calculated as half of the absolute difference between real and synthetic category proportions.

In [ ]:
def categorical_distribution_drift(real_data, synth_data, cat_cols):
    rows = []

    for col in cat_cols:
        real_dist = real_data[col].astype(str).value_counts(normalize=True, dropna=False)
        synth_dist = synth_data[col].astype(str).value_counts(normalize=True, dropna=False)

        all_categories = sorted(set(real_dist.index).union(set(synth_dist.index)))

        real_aligned = real_dist.reindex(all_categories, fill_value=0)
        synth_aligned = synth_dist.reindex(all_categories, fill_value=0)

        drift = 0.5 * np.abs(real_aligned - synth_aligned).sum()

        real_categories = set(real_dist.index)
        synth_categories = set(synth_dist.index)

        rows.append({
            "column": col,
            "real_category_count": len(real_categories),
            "synthetic_category_count": len(synth_categories),
            "category_coverage": len(real_categories.intersection(synth_categories)) / max(1, len(real_categories)),
            "new_synthetic_category_count": len(synth_categories - real_categories),
            "distribution_drift": drift,
            "max_category_proportion_diff": np.abs(real_aligned - synth_aligned).max()
        })

    return pd.DataFrame(rows).sort_values("distribution_drift", ascending=False)

cat_validation_mixed = categorical_distribution_drift(
    valid_eval_df,
    synthetic_mixed,
    categorical_cols
)

display(cat_validation_mixed)

display(cat_validation_mixed.describe(include="all"))

## Step 14: Numerical validation

Numerical validation checks whether the synthetic data preserves numerical distributions.

Metrics:

- Real mean vs synthetic mean
- Real median vs synthetic median
- Real standard deviation vs synthetic standard deviation
- Min/max comparison
- KS statistic
- PSI score

In [ ]:
def calculate_psi(real_values, synth_values, bins=10):
    real_values = pd.Series(real_values).dropna().astype(float)
    synth_values = pd.Series(synth_values).dropna().astype(float)

    if len(real_values) == 0 or len(synth_values) == 0:
        return np.nan

    # If the column is constant, PSI is zero when synthetic is also constant at same value.
    if real_values.nunique() <= 1:
        return 0.0 if synth_values.nunique() <= 1 and synth_values.iloc[0] == real_values.iloc[0] else np.nan

    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(real_values, quantiles))

    if len(edges) < 3:
        return np.nan

    real_counts, _ = np.histogram(real_values, bins=edges)
    synth_counts, _ = np.histogram(synth_values, bins=edges)

    real_pct = real_counts / max(1, real_counts.sum())
    synth_pct = synth_counts / max(1, synth_counts.sum())

    epsilon = 1e-6
    psi = np.sum((synth_pct - real_pct) * np.log((synth_pct + epsilon) / (real_pct + epsilon)))
    return float(psi)

def numerical_distribution_validation(real_data, synth_data, num_cols):
    rows = []

    for col in num_cols:
        real = pd.to_numeric(real_data[col], errors="coerce").dropna()
        synth = pd.to_numeric(synth_data[col], errors="coerce").dropna()

        if len(real) == 0 or len(synth) == 0:
            continue

        if SCIPY_AVAILABLE and real.nunique() > 1 and synth.nunique() > 1:
            ks_stat = ks_2samp(real, synth).statistic
        else:
            ks_stat = np.nan

        real_mean = real.mean()
        synth_mean = synth.mean()
        real_std = real.std()
        synth_std = synth.std()

        rows.append({
            "column": col,
            "real_mean": real_mean,
            "synthetic_mean": synth_mean,
            "mean_abs_diff": abs(real_mean - synth_mean),
            "real_median": real.median(),
            "synthetic_median": synth.median(),
            "median_abs_diff": abs(real.median() - synth.median()),
            "real_std": real_std,
            "synthetic_std": synth_std,
            "std_abs_diff": abs(real_std - synth_std),
            "real_min": real.min(),
            "synthetic_min": synth.min(),
            "real_max": real.max(),
            "synthetic_max": synth.max(),
            "ks_statistic": ks_stat,
            "psi": calculate_psi(real, synth)
        })

    return pd.DataFrame(rows).sort_values("ks_statistic", ascending=False, na_position="last")

num_validation_mixed = numerical_distribution_validation(
    valid_eval_df,
    synthetic_mixed,
    numerical_cols
)

display(num_validation_mixed)

display(num_validation_mixed.describe(include="all"))

## Step 15: Relationship validation

Single-column distributions are not enough.  
We also need to check whether relationships between columns are preserved.

This section checks:

1. Numerical correlation preservation
2. Categorical relationship preservation using Cramer's V
3. Categorical-numerical relationship preservation using group mean comparison

In [ ]:
def numerical_correlation_preservation(real_data, synth_data, num_cols):
    if len(num_cols) < 2:
        return pd.DataFrame()

    real_corr = real_data[num_cols].corr()
    synth_corr = synth_data[num_cols].corr()

    rows = []
    for c1, c2 in itertools.combinations(num_cols, 2):
        real_val = real_corr.loc[c1, c2]
        synth_val = synth_corr.loc[c1, c2]
        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_corr": real_val,
            "synthetic_corr": synth_val,
            "abs_corr_diff": abs(real_val - synth_val) if pd.notna(real_val) and pd.notna(synth_val) else np.nan
        })

    return pd.DataFrame(rows).sort_values("abs_corr_diff", ascending=False, na_position="last")

def cramers_v(x, y):
    contingency = pd.crosstab(x, y)

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return np.nan

    if not SCIPY_AVAILABLE:
        return np.nan

    chi2 = chi2_contingency(contingency, correction=False)[0]
    n = contingency.sum().sum()

    if n == 0:
        return np.nan

    r, k = contingency.shape
    return np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1)))

def categorical_relationship_preservation(real_data, synth_data, cat_cols):
    rows = []

    for c1, c2 in itertools.combinations(cat_cols, 2):
        real_v = cramers_v(real_data[c1].astype(str), real_data[c2].astype(str))
        synth_v = cramers_v(synth_data[c1].astype(str), synth_data[c2].astype(str))

        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_cramers_v": real_v,
            "synthetic_cramers_v": synth_v,
            "abs_cramers_v_diff": abs(real_v - synth_v) if pd.notna(real_v) and pd.notna(synth_v) else np.nan
        })

    return pd.DataFrame(rows).sort_values("abs_cramers_v_diff", ascending=False, na_position="last")

def categorical_numerical_relationship(real_data, synth_data, cat_cols, num_cols, max_categories=25):
    rows = []

    for cat in cat_cols:
        if real_data[cat].nunique(dropna=False) > max_categories:
            continue

        for num in num_cols:
            real_group = real_data.groupby(cat)[num].mean()
            synth_group = synth_data.groupby(cat)[num].mean()

            common_groups = sorted(set(real_group.index).intersection(set(synth_group.index)))

            if len(common_groups) == 0:
                continue

            real_values = real_group.reindex(common_groups)
            synth_values = synth_group.reindex(common_groups)

            scale = real_data[num].std()
            if pd.isna(scale) or scale == 0:
                scale = 1

            mean_group_diff = np.mean(np.abs(real_values - synth_values)) / scale

            rows.append({
                "categorical_column": cat,
                "numerical_column": num,
                "common_category_count": len(common_groups),
                "normalised_group_mean_diff": mean_group_diff
            })

    return pd.DataFrame(rows).sort_values("normalised_group_mean_diff", ascending=False, na_position="last")

num_corr_validation = numerical_correlation_preservation(
    valid_eval_df,
    synthetic_mixed,
    numerical_cols
)

cat_rel_validation = categorical_relationship_preservation(
    valid_eval_df,
    synthetic_mixed,
    categorical_cols
)

cat_num_rel_validation = categorical_numerical_relationship(
    valid_eval_df,
    synthetic_mixed,
    categorical_cols,
    numerical_cols
)

print("Numerical correlation preservation:")
display(num_corr_validation)

print("Categorical relationship preservation:")
display(cat_rel_validation)

print("Categorical-numerical relationship preservation:")
display(cat_num_rel_validation)

## Step 16: Privacy and duplicate-risk validation

This step checks whether the synthetic data simply copies too many real rows.

Metrics:

- Duplicate rate inside synthetic data
- Exact match rate between synthetic and training data
- Number of unique synthetic profiles
- Number of new synthetic profiles

In [ ]:
def row_hash_dataframe(data):
    # Use string representation so mixed dtypes can be compared consistently.
    return pd.util.hash_pandas_object(data.astype(str), index=False)

def privacy_duplicate_summary(train_data, synth_data):
    common_cols = [c for c in train_data.columns if c in synth_data.columns]

    train_hash = row_hash_dataframe(train_data[common_cols])
    synth_hash = row_hash_dataframe(synth_data[common_cols])

    synthetic_duplicate_rate = 1 - (synth_hash.nunique() / max(1, len(synth_hash)))
    exact_match_to_train_rate = synth_hash.isin(set(train_hash)).mean()

    return pd.DataFrame([{
        "synthetic_rows": len(synth_data),
        "synthetic_unique_rows": int(synth_hash.nunique()),
        "synthetic_duplicate_rate": synthetic_duplicate_rate,
        "exact_match_to_train_rate": exact_match_to_train_rate,
        "new_synthetic_profile_count": int((~synth_hash.isin(set(train_hash))).sum())
    }])

privacy_mixed = privacy_duplicate_summary(train_model_df, synthetic_mixed)

display(privacy_mixed)

## Step 17: Target validation

If the dataset contains a target column, the synthetic data should preserve target distribution.

This is especially important if synthetic data will be used for downstream ML experiments.

In [ ]:
if target_col is not None:
    target_validation = pd.concat(
        [
            train_model_df[target_col].value_counts(normalize=True, dropna=False).rename("train_rate"),
            valid_eval_df[target_col].value_counts(normalize=True, dropna=False).rename("validation_rate"),
            synthetic_mixed[target_col].value_counts(normalize=True, dropna=False).rename("synthetic_mixed_rate")
        ],
        axis=1
    ).fillna(0)

    display(target_validation)
else:
    print("No target column found. Skipping target validation.")

## Step 18: Utility validation

Utility validation checks whether synthetic data can support model training.

Comparison:

1. Train model on real training sample and test on validation data.
2. Train model on synthetic mixed data and test on the same validation data.

If synthetic-trained performance is close to real-trained performance, synthetic utility is stronger.

In [ ]:
def make_supervised_pipeline():
    transformers = []

    if len(categorical_cols) > 0:
        transformers.append((
            "cat",
            make_one_hot_encoder(),
            categorical_cols
        ))

    if len(numerical_cols) > 0:
        transformers.append((
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]),
            numerical_cols
        ))

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

    model = LogisticRegression(max_iter=1000, class_weight="balanced")

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

def utility_validation(real_train, synthetic_train, validation_data):
    if target_col is None:
        return pd.DataFrame([{"status": "skipped_no_target"}])

    y_real = real_train[target_col]
    y_synth = synthetic_train[target_col]
    y_valid = validation_data[target_col]

    if y_valid.nunique(dropna=True) != 2:
        return pd.DataFrame([{"status": "skipped_target_not_binary"}])

    if y_real.nunique(dropna=True) < 2 or y_synth.nunique(dropna=True) < 2:
        return pd.DataFrame([{"status": "skipped_one_training_set_has_single_target_class"}])

    X_real = real_train[feature_cols]
    X_synth = synthetic_train[feature_cols]
    X_valid = validation_data[feature_cols]

    rows = []

    for name, X_train, y_train in [
        ("real_train_to_validation", X_real, y_real),
        ("synthetic_mixed_train_to_validation", X_synth, y_synth)
    ]:
        pipeline = make_supervised_pipeline()
        pipeline.fit(X_train, y_train)

        proba = pipeline.predict_proba(X_valid)[:, 1]
        pred = (proba >= 0.5).astype(int)

        rows.append({
            "training_data": name,
            "auc": roc_auc_score(y_valid, proba),
            "accuracy": accuracy_score(y_valid, pred),
            "f1_score": f1_score(y_valid, pred)
        })

    return pd.DataFrame(rows)

utility_scorecard = utility_validation(
    train_model_df,
    synthetic_mixed,
    valid_eval_df
)

display(utility_scorecard)

## Step 19: Compare categorical-only, numerical-only, and mixed KNN outputs

This summary helps us understand which KNN view performs better.

The final recommended output is usually the **mixed categorical + numerical** version because it uses the most complete information.

In [ ]:
def single_run_scorecard(name, synthetic_data):
    cat_drift = categorical_distribution_drift(valid_eval_df, synthetic_data, categorical_cols)
    num_drift = numerical_distribution_validation(valid_eval_df, synthetic_data, numerical_cols)
    privacy = privacy_duplicate_summary(train_model_df, synthetic_data)

    row = {
        "run_name": name,
        "rows": len(synthetic_data),
        "columns": synthetic_data.shape[1],
        "mean_categorical_drift": cat_drift["distribution_drift"].mean() if len(cat_drift) else np.nan,
        "max_categorical_drift": cat_drift["distribution_drift"].max() if len(cat_drift) else np.nan,
        "mean_numeric_ks": num_drift["ks_statistic"].mean() if len(num_drift) else np.nan,
        "max_numeric_ks": num_drift["ks_statistic"].max() if len(num_drift) else np.nan,
        "mean_numeric_psi": num_drift["psi"].mean() if len(num_drift) else np.nan,
        "synthetic_duplicate_rate": privacy.loc[0, "synthetic_duplicate_rate"],
        "exact_match_to_train_rate": privacy.loc[0, "exact_match_to_train_rate"]
    }

    if target_col is not None:
        row["synthetic_target_rate_mean"] = pd.to_numeric(synthetic_data[target_col], errors="coerce").mean()

    return row

comparison_scorecard = pd.DataFrame([
    single_run_scorecard("categorical_only_knn", synthetic_categorical),
    single_run_scorecard("numerical_only_knn", synthetic_numerical),
    single_run_scorecard("mixed_categorical_numerical_knn", synthetic_mixed)
])

display(comparison_scorecard)

## Step 20: KNN sensitivity testing

This optional step tests multiple K values. It is turned off by default to keep the final notebook fast. Set `RUN_SENSITIVITY = True` in Step 1 to run it.

It shows whether the method is stable across different neighbour counts.

The sensitivity test still keeps all usable columns in the synthetic output.

In [ ]:
sensitivity_rows = []

if RUN_SENSITIVITY:
    matrix_cache = {
        "categorical": X_cat,
        "numerical": X_num,
        "mixed": X_mix
    }

    for mode_name, X_mode in matrix_cache.items():
        for k_value in SENSITIVITY_K_VALUES:
            synth_temp = generate_synthetic_knn(
                train_model_df,
                X_mode,
                mode_name=mode_name,
                n_rows=SENSITIVITY_SYNTHETIC_ROWS,
                k=k_value,
                random_state=RANDOM_STATE + 100 + k_value
            )

            row = single_run_scorecard(
                name=f"{mode_name}_k_{k_value}",
                synthetic_data=synth_temp
            )
            row["mode"] = mode_name
            row["k"] = k_value
            sensitivity_rows.append(row)

    sensitivity_scorecard = pd.DataFrame(sensitivity_rows)
    display(sensitivity_scorecard.sort_values(
        ["mean_categorical_drift", "mean_numeric_ks", "exact_match_to_train_rate"],
        ascending=[True, True, True]
    ))
else:
    sensitivity_scorecard = pd.DataFrame()
    print("Sensitivity testing skipped because RUN_SENSITIVITY = False.")

## Step 21: Final result summary

This section gives a direct summary of what happened in the experiment.

The final solution follows the updated requirement:

- Columns were **not** removed based on quality or usefulness.
- Constant columns were **kept**.
- High-cardinality columns were **kept**.
- Only fully missing/empty columns were dropped.
- Both categorical and numerical columns were used.
- No multiple result files were generated.

In [ ]:
summary_lines = []

summary_lines.append(f"Original dataset shape: {raw_df.shape}")
summary_lines.append(f"Final usable dataset shape after dropping fully empty columns only: {df.shape}")
summary_lines.append(f"Fully empty/missing columns dropped: {len(empty_columns)}")
summary_lines.append(f"Dropped columns: {empty_columns}")
summary_lines.append(f"Categorical feature columns kept: {len(categorical_cols)}")
summary_lines.append(f"Numerical feature columns kept: {len(numerical_cols)}")
summary_lines.append(f"Target column: {target_col}")
summary_lines.append(f"Synthetic mixed dataset shape: {synthetic_mixed.shape}")

if len(cat_validation_mixed) > 0:
    summary_lines.append(f"Mean categorical distribution drift: {cat_validation_mixed['distribution_drift'].mean():.6f}")
    summary_lines.append(f"Max categorical distribution drift: {cat_validation_mixed['distribution_drift'].max():.6f}")

if len(num_validation_mixed) > 0:
    summary_lines.append(f"Mean numerical KS statistic: {num_validation_mixed['ks_statistic'].mean():.6f}")
    summary_lines.append(f"Max numerical KS statistic: {num_validation_mixed['ks_statistic'].max():.6f}")

summary_lines.append(f"Synthetic duplicate rate: {privacy_mixed.loc[0, 'synthetic_duplicate_rate']:.6f}")
summary_lines.append(f"Exact match to train rate: {privacy_mixed.loc[0, 'exact_match_to_train_rate']:.6f}")

display(Markdown("\n".join([f"- {line}" for line in summary_lines])))

print("\nFinal synthetic mixed sample:")
display(synthetic_mixed.head(10))

## Step 22: Recommended next actions

1. Review dropped columns and confirm they were truly empty in the source data.
2. Review constant columns with the business team. They are kept, but they do not help distance calculation.
3. Compare categorical-only, numerical-only, and mixed KNN scorecards.
4. Prefer the mixed KNN run if it gives the best balance of:
   - low categorical drift
   - low numerical KS/PSI
   - acceptable privacy risk
   - acceptable model utility
5. If privacy risk is high, increase K, add controlled noise to numerical interpolation, or reduce direct neighbour copying.
6. If numerical drift is high, tune categorical/numerical weights in the mixed distance.
7. If categorical drift is high, use weighted category sampling or category-level smoothing.